Use MCMC & Zeus to extract parameter values \
Based off of the Zeus tutorial

In [26]:
import cosmo_calc
import importlib
import zeus21

In [27]:
# Get cosmological parameters, class to construct the HMF from Zeus
CosmoParams_input = zeus21.Cosmo_Parameters_Input(zmin_CLASS=0.0)
CosmoParams,ClassyCosmo, CorrFclass ,HMFintclass =  zeus21.cosmo_wrapper(CosmoParams_input)

<span style = 'color:red'> Here's where I need to add the time evolution <span>

In [28]:
def param_wrapper(paramvector, CosmoParams):
    """
    Puts paramvector into a format that Zeus can read
    paramvector [1darray]: log10eps, log10Mc, alpha, beta
    """
    
    log10epsstar, log10Mcstar, alphastar, betastar = paramvector
    astroparams = zeus21.Astro_Parameters(CosmoParams,epsstar=10**log10epsstar, Mc=10**log10Mcstar,alphastar=alphastar, betastar=betastar) 
    
    return astroparams

In [29]:
def UVLF_wrapper(zcenters, zwidths, MUVcenters, MUVwidths, paramvector, CosmoParams):
    'Computes and returns the UVLF at z=zcenters, with width zwidths, in bins centered at MUVcenters with width MUVwidths'
    'Output is PhiUV, depends on astro parameter inputs paramvector'
    astroparams = param_wrapper(paramvector, CosmoParams)
    UVLFs_std = zeus21.UVLFs.UVLF_binned(astroparams,CosmoParams,HMFintclass,zcenters,zwidths,MUVcenters,MUVwidths)
    return UVLFs_std

In [42]:
def log_prior(paramvector):
    """
    Modify the numbers here to change the prior
    """

    log10epsstar, log10Mcstar, alphastar, betastar = paramvector
    
    if (log10epsstar>1.0 or log10epsstar<-5.0):
        return -np.inf  
    elif (log10Mcstar>15 or log10Mcstar<10):
        return -np.inf  
    elif (alphastar<0.0 or alphastar>3.0):
        return -np.inf
    elif (betastar>0 or betastar<-3.0):
        return -np.inf
    else:
        return 0.0
  
    
def log_like(paramvector, datavector, CosmoParams):
    'Sample flat in eps, alph, beta, and log10Mc. datavector has all the z and UVLF data'
    
    #now bin it appropriately at each z -- data part
    loglike_curr = 0.0

    for dataarrayz in datavector: # Add the log likelihoods together for each redshift. The log likelihood is just a sum over all the points
        # anyways, so this makes sense
        #datHSTz4=[3.8, mags_z4,phi_z4,err_z4,errx_z4]
        zdat = dataarrayz[0]
        zerr = dataarrayz[1]
        xdat = dataarrayz[2]
        ydat = dataarrayz[3]
        yerr = dataarrayz[4] 
        xerr = dataarrayz[5]                
        #izus = np.argmin(np.abs(z0list - zdat))      
        
        yerr = np.fmax(yerr, ydat*MINRELERROR) # Make sure error bars aren't any smaller than the relative error set above
                    
        uvlftheory = UVLF_wrapper(zdat,zerr,xdat,xerr, paramvector, CosmoParams)
        
        loglike_curr += -np.sum( (ydat - uvlftheory)**2/(2.0 * yerr**2) ) #assumed Gaussian, to be revisited.
    
    return loglike_curr


def log_prob(paramvector, datavector, CosmoParams):
    
    lprior=log_prior(paramvector)
    if (lprior > -np.inf): #only run if in prior range. avoid weird behavior for negative logs etc
        lpost=log_like(paramvector, datavector, CosmoParams)
    else:
        lpost = 0.0 #doesn't matter, added to -inf
    return lprior + lpost

In [31]:
datBouw21 = np.loadtxt('/Users/eb35267/Desktop/research/data/Bouwens2021_GAL.txt',  skiprows=2, unpack=True)
redshiftsBouw21 = np.unique(datBouw21[0])
dredshiftsBouw21 = np.ones_like(redshiftsBouw21)/2. #approximate, there are true window functions to use

datHST = [] 
#format is     zdat = data[0] zerr = data[1] xdat = data[2],  ydat = data[3]  yerr = data[4]  xerr = data[5] 

for iz,z in enumerate(redshiftsBouw21): # Create a bunch of arrays that contain all of the data listed above. One for each redshift
    print(z)
    zlistindex = datBouw21[0] == 1.0*z
    datarr = [z,dredshiftsBouw21[iz], datBouw21[1][zlistindex], datBouw21[3][zlistindex], datBouw21[4][zlistindex], datBouw21[2][zlistindex]]
    datHST= datHST + [datarr]

4.0
5.0
6.0
7.0
8.0
9.0
10.0


In [55]:
params1 = np.array([-1, 12, 0.6, 0.5])
ndim = len(params1)
nwalkers = 2*ndim
N = 100

p0 = 2*(1-np.random.rand(nwalkers, ndim))+params1 # Start each of the random walkers in a slightly different place 

In [56]:
p0

array([[ 0.36303638, 13.96162874,  1.04675936,  1.74069505],
       [ 0.89826186, 13.88659558,  1.99406297,  1.4936028 ],
       [-0.85886882, 13.82213935,  2.54264816,  2.42572818],
       [-0.18139665, 12.76199071,  1.67574249,  2.28754909],
       [ 0.8432725 , 12.65999288,  1.06416618,  1.73807703],
       [ 0.20488895, 13.42536139,  2.12423039,  1.3024362 ],
       [ 0.66688089, 13.24907674,  1.09289228,  1.49932078],
       [ 0.87686462, 13.66259753,  1.08476339,  1.40326895]])

In [35]:
(1 + np.random.rand(nwalkers, ndim) * 2)

array([[2.26916869, 2.37802211, 2.34200122, 1.45389698],
       [1.53612612, 2.92482291, 1.85618851, 1.53733766],
       [1.46298436, 2.38433555, 2.92755703, 1.77540138],
       [1.03802607, 2.26343481, 2.72565662, 1.9879761 ],
       [1.09575369, 1.30028597, 2.0238033 , 1.85557497],
       [1.63758227, 1.59522672, 1.07532329, 1.51965887],
       [2.09756916, 2.29545219, 1.22425893, 1.37312008],
       [1.85861259, 2.904262  , 2.018727  , 2.62535898]])

In [ ]:
#trim p0 of walkers that are outside of prior range
for ip,pval in enumerate(p0):
    priorval = log_prior(p0[ip]) 
    print(priorval)
    while(priorval<0.0):
        p0[ip] = (1 + np.random.rand(ndim) * 2) * params1
        priorval = log_prior(p0[ip])   

In [ ]:
sampler_hst = emcee.EnsembleSampler(nwalkers, ndim, log_prob, args=[datHST, CosmoParams])
state_hst = sampler_hst.run_mcmc(p0, N)
sampler_hst.reset()      